In [1]:
import re
import csv
import numpy as np
import pandas as pd
from pathlib import Path


# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"



# =========================
# PARAMS À ADAPTER
# =========================
VIDEO_DIR = DATA_DIR / "video"   # ou "Vidéo" selon le vrai nom exact du dossier

EXCEL_DIR = DATA_DIR / "Excels_code"

ROOT_MP = DATA_DIR / "video"

ROOT_VICON = DATA_DIR / "VICON_CSV"       # racine où sont tes dossiers D01/P1/SEATED/... avec les *_pose.xlsx

FPS_VICON = 100.0
FPS_MP    = 30.0

# =========================
# 1) Lecture CSV Vicon (ton format)
# =========================
def read_vicon_csv(csv_path: Path) -> pd.DataFrame:
    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        header_lines = [next(reader) for _ in range(5)]

    marker_row = header_lines[2]   # ligne 3
    axis_row   = header_lines[3]   # ligne 4

    n = max(len(marker_row), len(axis_row))
    marker_row += [""] * (n - len(marker_row))
    axis_row   += [""] * (n - len(axis_row))

    # forward-fill noms marqueurs
    filled = []
    last = ""
    for m in marker_row:
        m = (m or "").strip()
        if m == "":
            filled.append(last)
        else:
            last = m
            filled.append(last)

    # noms plats
    colnames = []
    for m, a in zip(filled, axis_row):
        m = (m or "").strip()
        a = (a or "").strip()

        if a in ["Frame", "Sub Frame"]:
            colnames.append(a)
        elif a in ["X", "Y", "Z"]:
            colnames.append(f"{m}_{a}")
        else:
            colnames.append(m if m else a)

    df = pd.read_csv(csv_path, skiprows=5, header=None, names=colnames, engine="python")
    df = df.dropna(axis=1, how="all")
    return df

def find_xyz_cols(cols, token):
    """
    token: 'poignet_D', '2epaule_G', etc.
    match: 'Patient 1:poignet_D_X' ou 'D12:epaule_G_X' etc.
    """
    pat = re.compile(rf"(?:^|:)\s*{re.escape(token)}_([XYZ])\b", re.IGNORECASE)
    found = {}
    for c in cols:
        c2 = c.replace(" ", "")
        m = pat.search(c2)
        if m:
            axis = m.group(1).upper()
            if axis not in found or len(c) < len(found[axis]):
                found[axis] = c
    return found.get("X"), found.get("Y"), found.get("Z")

# =========================
# 2) Trouver les pose.xlsx MediaPipe (P1/P2)
# =========================
def parse_vicon_name(csv_name: str):
    """
    Ex: SEATEDD01.csv -> condition=SEATED, dyad=D01
        SEMID12.csv    -> condition=SEMI,   dyad=D12   (on standardise SEMI)
    """
    s = csv_name.upper().replace(".CSV","")
    # condition au début
    if s.startswith("SEATED"):
        cond = "SEATED"
    elif s.startswith("STANDING"):
        cond = "STANDING"
    elif s.startswith("SEMI"):
        cond = "SEMI"
    else:
        cond = None

    m = re.search(r"(D\d+)", s)
    dyad = m.group(1) if m else None
    return cond, dyad

def find_mp_pose_xlsx(dyad: str, cond: str, pid: str, root: Path):
    """
    Cherche un *_pose.xlsx dans l'arborescence.
    Hypothèse : tes fichiers sont dans .../D01/P1/SEATED/..._pose.xlsx
    On cherche dyad + pid + cond dans le chemin.
    """
    dyad_num = dyad.replace("D","")  # "01"
    # patterns permissifs
    candidates = []
    for f in root.rglob("*_pose.xlsx"):
        p = str(f).upper()
        if (f"/D{dyad_num}/" in p or f"\\D{dyad_num}\\" in p or dyad in p):
            if f"/{pid}/" in p or f"\\{pid}\\" in p:
                if f"/{cond}/" in p or f"\\{cond}\\" in p or cond in p:
                    candidates.append(f)
    if candidates:
        # prend le plus court / le plus "direct"
        candidates.sort(key=lambda x: len(str(x)))
        return candidates[0]
    return None

# =========================
# 3) Amplitude poignet↔centre épaules
# =========================
def amplitude_distance_vicon_plane(arr_w, arr_ls, arr_rs, plane="XZ"):
    """
    Vicon est (n,3). On projette en 2D selon le plan choisi :
    XY, XZ ou YZ.
    Retourne min, max, peak-to-peak.
    """
    plane_indices = {
        "XY": [0, 1],
        "XZ": [0, 2],
        "YZ": [1, 2],
    }

    idx = plane_indices[plane]

    w2  = arr_w[:, idx]
    ls2 = arr_ls[:, idx]
    rs2 = arr_rs[:, idx]

    C = (ls2 + rs2) / 2.0
    Wrel = w2 - C
    d = np.sqrt((Wrel**2).sum(axis=1))
    d = d[np.isfinite(d)]

    if d.size == 0:
        return np.nan, np.nan, np.nan

    return float(d.min()), float(d.max()), float(d.max() - d.min())

def mp_arrays_from_pose(df_pose, wrist_side="RIGHT"):
    """
    wrist_side: 'RIGHT' ou 'LEFT'
    MediaPipe Pose colonnes attendues: RIGHT_WRIST_x, RIGHT_WRIST_y, RIGHT_SHOULDER_x, ...
    """
    w = df_pose[[f"{wrist_side}_WRIST_x", f"{wrist_side}_WRIST_y"]].to_numpy(float)
    ls = df_pose[["LEFT_SHOULDER_x","LEFT_SHOULDER_y"]].to_numpy(float)
    rs = df_pose[["RIGHT_SHOULDER_x","RIGHT_SHOULDER_y"]].to_numpy(float)
    return w, ls, rs

def vicon_arrays(df_v, cols, pid="P1", wrist_side="D"):
    """
    pid=P1 ou P2
    wrist_side 'D' ou 'G' pour poignet_D / poignet_G
    épaules: epaule_D / epaule_G (ou 2epaule_D / 2epaule_G)
    """
    if pid == "P1":
        wtoken = f"poignet_{wrist_side}"
        shD = "epaule_D"
        shG = "epaule_G"
    else:
        wtoken = f"2poignet_{wrist_side}"
        shD = "2epaule_D"
        shG = "2epaule_G"

    wX,wY,wZ   = find_xyz_cols(cols, wtoken)
    gX,gY,gZ   = find_xyz_cols(cols, shG)
    dX,dY,dZ   = find_xyz_cols(cols, shD)

    if None in [wX,wY,wZ,gX,gY,gZ,dX,dY,dZ]:
        return None

    w  = df_v[[wX,wY,wZ]].to_numpy(float)
    ls = df_v[[gX,gY,gZ]].to_numpy(float)  # épaule gauche
    rs = df_v[[dX,dY,dZ]].to_numpy(float)  # épaule droite
    return w, ls, rs

# =========================
# 4) Matching frames (sans interpolation)
# =========================
def map_mp_to_vicon_idx(n_mp, n_vicon, fps_mp=30.0, fps_vicon=100.0):
    idx = np.rint(np.arange(n_mp) * (fps_vicon / fps_mp)).astype(int)
    idx = np.clip(idx, 0, n_vicon-1)
    return idx

# =========================
# 5) Amplitude distance poignet↔centre épaules pour MP (2D) et Vicon (3D projeté en XZ)
# =========================
def amplitude_distance(w, ls, rs):
    """
    MediaPipe en 2D : distance poignet-centre épaules.
    Retourne min, max, peak-to-peak.
    """
    C = (ls + rs) / 2.0
    Wrel = w - C
    d = np.sqrt((Wrel**2).sum(axis=1))
    d = d[np.isfinite(d)]

    if d.size == 0:
        return np.nan, np.nan, np.nan

    return float(d.min()), float(d.max()), float(d.max() - d.min())
# =========================
# 5) Batch sur tous les CSV Vicon
# =========================
rows = []

for csv_path in sorted(ROOT_VICON.rglob("*.csv")):
    cond, dyad = parse_vicon_name(csv_path.name)
    if cond is None or dyad is None:
        continue

    # Vicon df
    df_v = read_vicon_csv(csv_path)
    cols = list(df_v.columns)
    n_v = len(df_v)

    # MP pose: P1/P2
    mp_p1 = find_mp_pose_xlsx(dyad, cond, "P1", ROOT_MP)
    mp_p2 = find_mp_pose_xlsx(dyad, cond, "P2", ROOT_MP)

    base = {
        "vicon_csv": str(csv_path),
        "file": csv_path.name,
        "dyad": dyad,
        "condition": cond,
        "mp_pose_p1": str(mp_p1) if mp_p1 else None,
        "mp_pose_p2": str(mp_p2) if mp_p2 else None,
    }

    for pid, mp_path in [("P1", mp_p1), ("P2", mp_p2)]:
        out = base.copy()
        out["pid"] = pid

        if mp_path is None:
            out["error"] = "missing mp pose.xlsx"
            rows.append(out)
            continue

        df_mp = pd.read_excel(mp_path)
        n_mp = len(df_mp)

        # mapping MP -> Vicon
        idx_v = map_mp_to_vicon_idx(n_mp, n_v, FPS_MP, FPS_VICON)

        # --- MP amplitude (2D) ---
        try:
            # Right/Left wrists
            wR, ls2, rs2 = mp_arrays_from_pose(df_mp, "RIGHT")
            wL, ls2b, rs2b = mp_arrays_from_pose(df_mp, "LEFT")  # mêmes épaules
            mp_R = amplitude_distance(wR, ls2, rs2)
            mp_L = amplitude_distance(wL, ls2, rs2)

            out["MP_ampR_min"] = mp_R[0]; out["MP_ampR_max"] = mp_R[1]; out["MP_ampR_pp"] = mp_R[2]
            out["MP_ampL_min"] = mp_L[0]; out["MP_ampL_max"] = mp_L[1]; out["MP_ampL_pp"] = mp_L[2]
            out["MP_ampWRISTS_pp_sum"] = out["MP_ampR_pp"] + out["MP_ampL_pp"]
        except Exception as e:
            out["error"] = f"mp compute fail: {e}"
            rows.append(out)
            continue

        # --- Vicon amplitude (3D), évaluée aux frames correspondantes (timeline MP) ---
        # Matching sides: pour être cohérent, on compare:
        # - RIGHT wrist MP ~ poignet_D côté Vicon (si ton naming D=right)
        # - LEFT  wrist MP ~ poignet_G côté Vicon
        # (Si jamais D/G inversé chez toi, tu inverseras ici)
        vk_R = vicon_arrays(df_v, cols, pid=pid, wrist_side="D")
        vk_L = vicon_arrays(df_v, cols, pid=pid, wrist_side="G")

        if vk_R is None or vk_L is None:
            out["error"] = "missing vicon wrist/shoulders columns"
            rows.append(out)
            continue

        wR3, ls3, rs3 = vk_R
        wL3, ls3b, rs3b = vk_L

        # on prend les frames Vicon correspondant aux frames MP
        wR3 = wR3[idx_v]; wL3 = wL3[idx_v]
        ls3 = ls3[idx_v]; rs3 = rs3[idx_v]

        for plane in ["XY", "XZ", "YZ"]:
            v_R = amplitude_distance_vicon_plane(wR3, ls3, rs3, plane=plane)
            v_L = amplitude_distance_vicon_plane(wL3, ls3, rs3, plane=plane)

            out[f"VICON_{plane}_ampR_min_mm"] = v_R[0]
            out[f"VICON_{plane}_ampR_max_mm"] = v_R[1]
            out[f"VICON_{plane}_ampR_pp_mm"] = v_R[2]

            out[f"VICON_{plane}_ampL_min_mm"] = v_L[0]
            out[f"VICON_{plane}_ampL_max_mm"] = v_L[1]
            out[f"VICON_{plane}_ampL_pp_mm"] = v_L[2]

            out[f"VICON_{plane}_ampWRISTS_pp_sum_mm"] = (
                out[f"VICON_{plane}_ampR_pp_mm"] +
                out[f"VICON_{plane}_ampL_pp_mm"]
            )

        rows.append(out)

df_amp = pd.DataFrame(rows)

out_path = EXCEL_DIR / "AMP_wrists_MP_vs_VICON_noInterp.xlsx"
df_amp.to_excel(out_path, index=False)
print("✅ Saved:", out_path)
print(df_amp.head())

✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/AMP_wrists_MP_vs_VICON_noInterp.xlsx
                                           vicon_csv           file dyad  \
0  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  SEATEDD01.csv  D01   
1  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  SEATEDD01.csv  D01   
2  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  SEATEDD02.csv  D02   
3  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  SEATEDD02.csv  D02   
4  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VI...  SEATEDD03.csv  D03   

  condition                                         mp_pose_p1  \
0    SEATED  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/vi...   
1    SEATED  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/vi...   
2    SEATED  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/vi...   
3    SEATED  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/vi...   
4    SEATED  /Users/matysprecloux/Desktop/SYNCOGEST/DATA/vi...   

                                          mp

In [4]:
import pandas as pd
import numpy as np
import re

AMP_PATH = "/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/AMP_wrists_MP_vs_VICON_noInterp.xlsx"
MP_SW_PATH = "/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/mediapipe_QDM_filtered_and_SW.xlsx"
VICON_PATH = "/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/vicon_QDM_wrists_head_normByShoulders.xlsx"

OUT_PATH = "/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/amplitude_WRISTS_MP_scaled_to_mm.xlsx"

def clean_base(x):
    x = str(x)
    x = x.replace(".csv","").replace(".xlsx","")
    x = x.strip()
    return x

# ----------------------------
# 1) Load
# ----------------------------
amp = pd.read_excel(AMP_PATH)
mp  = pd.read_excel(MP_SW_PATH)
vi  = pd.read_excel(VICON_PATH)

# ----------------------------
# 2) AMP: video_base + pid
# ----------------------------
# On suppose que amp a déjà une colonne pid (P1/P2) et une colonne vidéo type file/csv/video_id
vid_col = None
for c in ["video_id","video_base","video_base_norm","file","csv"]:
    if c in amp.columns:
        vid_col = c
        break
if vid_col is None:
    raise ValueError("Amplitude: pas trouvé de colonne vidéo (video_id/file/csv/...).")

pid_col = None
for c in ["pid","P","participant","subject"]:
    if c in amp.columns:
        pid_col = c
        break
if pid_col is None:
    raise ValueError("Amplitude: pas trouvé de colonne pid.")

amp = amp.rename(columns={vid_col:"video_raw", pid_col:"pid"})
amp["video_base"] = amp["video_raw"].apply(clean_base)

# ----------------------------
# 3) MediaPipe: extraire pid depuis video_id (SEATEDD01_P1)
# ----------------------------
mp = mp.copy()
mp["pid"] = mp["video_id"].astype(str).str.extract(r"_(P\d)$")[0]   # P1/P2
mp["video_base"] = mp["video_id"].astype(str).str.replace(r"_(P\d)$","", regex=True)

# Normaliser aussi (au cas où)
mp["video_base"] = mp["video_base"].apply(clean_base)

# On garde SW_mp_median
mp_sw = mp[["video_base","pid","SW_mp_median"]].copy()
mp_sw = mp_sw.rename(columns={"SW_mp_median":"SW_mp"})

# ----------------------------
# 4) Vicon: passer P1/P2 shoulder width en format long
# ----------------------------
vi = vi.copy()
vi["video_base"] = vi["video_id"].apply(clean_base)

vi_long = vi[["video_base","P1_shoulder_width_median_mm","P2_shoulder_width_median_mm"]].melt(
    id_vars=["video_base"],
    value_vars=["P1_shoulder_width_median_mm","P2_shoulder_width_median_mm"],
    var_name="pid",
    value_name="SW_vicon_mm"
)

vi_long["pid"] = vi_long["pid"].str.extract(r"^(P\d)")[0]  # P1/P2

# ----------------------------
# 5) Merge + scale
# ----------------------------
df = amp.merge(mp_sw, on=["video_base","pid"], how="left")
df = df.merge(vi_long, on=["video_base","pid"], how="left")

print("Missing SW_mp:", df["SW_mp"].isna().sum(), "/", len(df))
print("Missing SW_vicon_mm:", df["SW_vicon_mm"].isna().sum(), "/", len(df))
print(df[["video_base","pid","SW_mp","SW_vicon_mm"]].head(12))

df["scale_mp_to_mm"] = df["SW_vicon_mm"] / df["SW_mp"]

# ----------------------------
# 6) Convert MP amplitude -> mm
# ----------------------------
mp_amp_cols = [c for c in df.columns if c.startswith("MP_amp") and not c.endswith("_mm")]

for c in mp_amp_cols:
    df[c + "_mm"] = df[c].astype(float) * df["scale_mp_to_mm"].astype(float)

# ----------------------------
# 7) Save
# ----------------------------
df.to_excel(OUT_PATH, index=False)
print("✅ Saved:", OUT_PATH)

Missing SW_mp: 0 / 120
Missing SW_vicon_mm: 0 / 120
   video_base pid     SW_mp  SW_vicon_mm
0   SEATEDD01  P1  0.067320   281.600411
1   SEATEDD01  P2  0.076967   305.506314
2   SEATEDD02  P1  0.071646   286.055610
3   SEATEDD02  P2  0.093298   305.121583
4   SEATEDD03  P1  0.074514   289.523437
5   SEATEDD03  P2  0.065281   262.377859
6   SEATEDD04  P1  0.088945   284.671404
7   SEATEDD04  P2  0.087887   348.361032
8   SEATEDD05  P1  0.076943   263.954779
9   SEATEDD05  P2  0.099341   361.325059
10  SEATEDD06  P1  0.097813   306.163399
11  SEATEDD06  P2  0.076331   306.555464
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/amplitude_WRISTS_MP_scaled_to_mm.xlsx


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# PARAMS
# =========================
ROOT = Path("/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/")

# =========================
# FONCTION AMPLITUDE
# =========================
def compute_amplitude(w, ls, rs):
    """
    Distance poignet ↔ centre des épaules (2D)
    """
    C = (ls + rs) / 2.0
    Wrel = w - C
    d = np.sqrt((Wrel**2).sum(axis=1))
    d = d[np.isfinite(d)]

    if len(d) == 0:
        return np.nan, np.nan, np.nan

    return d.min(), d.max(), d.max() - d.min()

# =========================
# LOOP SUR TOUS LES FICHIERS
# =========================
rows = []

for file in ROOT.rglob("*_pose.xlsx"):
    df = pd.read_excel(file)

    try:
        # RIGHT WRIST
        wR = df[["RIGHT_WRIST_x", "RIGHT_WRIST_y"]].to_numpy(float)

        # LEFT WRIST
        wL = df[["LEFT_WRIST_x", "LEFT_WRIST_y"]].to_numpy(float)

        # épaules
        ls = df[["LEFT_SHOULDER_x", "LEFT_SHOULDER_y"]].to_numpy(float)
        rs = df[["RIGHT_SHOULDER_x", "RIGHT_SHOULDER_y"]].to_numpy(float)

        # amplitude
        min_R, max_R, amp_R = compute_amplitude(wR, ls, rs)
        min_L, max_L, amp_L = compute_amplitude(wL, ls, rs)

        rows.append({
            "file": file.name,
            "MP_ampR_min": min_R,
            "MP_ampR_max": max_R,
            "MP_ampR_pp": amp_R,
            "MP_ampL_min": min_L,
            "MP_ampL_max": max_L,
            "MP_ampL_pp": amp_L,
            "MP_amp_WRISTS_sum": amp_R + amp_L
        })

    except Exception as e:
        rows.append({
            "file": file.name,
            "error": str(e)
        })

# =========================
# SAVE
# =========================
df_out = pd.DataFrame(rows)

out_path = ROOT / "MP_amplitudes_wrists.xlsx"
df_out.to_excel(out_path, index=False)

print("✅ Saved:", out_path)
print(df_out.head())

✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/MP_amplitudes_wrists.xlsx
                       file  MP_ampR_min  MP_ampR_max  MP_ampR_pp  \
0    SEATEDD05_P1_pose.xlsx     0.043524     0.227757    0.184233   
1      SEMID05_P1_pose.xlsx     0.013022     0.249118    0.236096   
2  STANDINGD05_P1_pose.xlsx     0.044812     0.270083    0.225271   
3    SEATEDD05_P2_pose.xlsx     0.189202     0.205601    0.016399   
4      SEMID05_P2_pose.xlsx     0.241180     0.271779    0.030600   

   MP_ampL_min  MP_ampL_max  MP_ampL_pp  MP_amp_WRISTS_sum  
0     0.035182     0.225812    0.190630           0.374863  
1     0.163930     0.241793    0.077864           0.313960  
2     0.042977     0.247561    0.204584           0.429855  
3     0.026355     0.195719    0.169364           0.185763  
4     0.203964     0.259899    0.055934           0.086534  
